In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [5]:
# load data
data = pd.read_csv("/content/drive/MyDrive/dataset/HAM10000_metadata.csv")

In [6]:
import os
import glob

image_paths = {}
folders = ["/content/drive/MyDrive/dataset/HAM10000_images_part_1", "/content/drive/MyDrive/dataset/HAM10000_images_part_2"]
for folder in folders:
    for img_path in glob.glob(os.path.join(folder, "*.jpg")):
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        image_paths[image_id] = img_path

print("Total images:", len(image_paths))

Total images: 10046


In [7]:
data["image_path"] = data["image_id"].map(image_paths)
print(data["image_path"].isnull().sum())

0


In [8]:
# tarin validation test-split
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    stratify=data["dx"],
    random_state=42
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dx"],
    random_state=42
)

In [9]:
print("Training Dataset :", train_df.shape)
print("Validation Dataset :", val_df.shape)
print("Testing Dataset :", test_df.shape)

Training Dataset : (7010, 8)
Validation Dataset : (1502, 8)
Testing Dataset : (1503, 8)


In [10]:
# handli class imbalance
from sklearn.utils.class_weight import compute_class_weight


classes = np.unique(train_df["dx"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["dx"]
)

class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [1]:
EfficientNetB0_data=[]
EfficientNetB0_data.append({
    "Model": "EfficientNetB0",
    "Train Accuracy": 51.68,
    "Validation Accuracy": 56.26,
    "Test Accuracy": 54.49,
    "Train Loss": 1.3004,
    "Validation Loss": 1.2470,
    "Test Loss": 1.2937
})

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,
    height_shift_range=0.1,

    horizontal_flip=True,
    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]
)
# Validation Generator
val_datagen = ImageDataGenerator(
    rescale=1./255
)
# Test Generator
test_datagen = ImageDataGenerator(
    rescale=1./255
)


In [13]:
# create train  pipeline
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)
# create validatation pipeline
val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)
# create test pipeline
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,

    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)




Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [13]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

In [14]:
# create new data generator
train_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2]
)

val_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen_eff = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [15]:
train_generator_eff = train_datagen_eff.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=True
)

val_generator_eff = val_datagen_eff.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

test_generator_eff = test_datagen_eff.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [18]:
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras import Model
from tensorflow.keras.applications import EfficientNetB0


In [19]:
from tensorflow.keras.layers import GlobalAveragePooling2D

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(7, activation="softmax")(x)

efficientnet_model = Model(
    inputs=base_model.input,
    outputs=output,
    name="EfficientNetB0"
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [20]:
from tensorflow.keras.callbacks import EarlyStopping

In [21]:
from tensorflow.keras.callbacks import EarlyStopping
# callback
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [22]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

# create the scheduler
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss", factor=0.2, patience=2,
    min_lr=1e-6, verbose=1
)

In [21]:
efficientnet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [24]:
history_efficientnet = efficientnet_model.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 408s 2s/step - accuracy: 0.3984 - loss: 1.6256 - val_accuracy: 0.5439 - val_loss: 1.3929 - learning_rate: 1.0000e-04
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 183s 833ms/step - accuracy: 0.4725 - loss: 1.4631 - val_accuracy: 0.5646 - val_loss: 1.3077 - learning_rate: 1.0000e-04
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 853ms/step - accuracy: 0.4850 - loss: 1.3716 - val_accuracy: 0.5679 - val_loss: 1.2329 - learning_rate: 1.0000e-04
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 175s 795ms/step - accuracy: 0.5043 - loss: 1.3024 - val_accuracy: 0.5646 - val_loss: 1.1939 - learning_rate: 1.0000e-04
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 169s 770ms/step - accuracy: 0.5140 - loss: 1.2602 - val_accuracy: 0.5999 - val_loss: 1.0835 - learning_rate: 1.0000e-04
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 193s 879ms/step - accuracy: 0.5317 - loss: 1.2429 - val_accuracy: 0.5539 - val_loss: 1.1894 - learning_rate: 1.0000e-04
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [25]:
train_loss, train_accuracy = efficientnet_model.evaluate(train_generator_eff)
val_loss, val_accuracy = efficientnet_model.evaluate(val_generator_eff)
test_loss, test_accuracy = efficientnet_model.evaluate(test_generator_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 168s 762ms/step - accuracy: 0.6195 - loss: 1.1024
47/47 ━━━━━━━━━━━━━━━━━━━━ 15s 324ms/step - accuracy: 0.5999 - loss: 1.0835
47/47 ━━━━━━━━━━━━━━━━━━━━ 127s 3s/step - accuracy: 0.6055 - loss: 1.1392


In [50]:
import os
efficientnet_model.save("/content/drive/MyDrive/models3/efficientnet_model.keras")

In [2]:
EfficientNetB0_data.append({
    "Model": "EfficientNetB0 ES_LR",
    "Train Accuracy": 61.95,
    "Validation Accuracy": 59.99,
    "Test Accuracy": 60.55,
    "Train Loss": 1.1024,
    "Validation Loss": 1.0835,
    "Test Loss": 1.1392
})

In [ ]:
import torch
print(torch.cuda.is_available())

**SGD**

In [28]:
efficientnet_sgd = Model(
    inputs=base_model.input,
    outputs=output,
    name="EfficientNetB0"
)

In [29]:
from tensorflow.keras.optimizers import SGD

efficientnet_sgd.compile(
    optimizer=SGD( learning_rate=0.01, momentum=0.9, nesterov=True),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [30]:
history_efficientnet_sgd = efficientnet_sgd.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 232s 952ms/step - accuracy: 0.4067 - loss: 1.5606 - val_accuracy: 0.6132 - val_loss: 1.0996 - learning_rate: 0.0100
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 779ms/step - accuracy: 0.4226 - loss: 1.4395
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0019999999552965165.
220/220 ━━━━━━━━━━━━━━━━━━━━ 187s 848ms/step - accuracy: 0.4200 - loss: 1.4880 - val_accuracy: 0.5173 - val_loss: 1.1112 - learning_rate: 0.0100
Epoch 2: early stopping
Restoring model weights from the end of the best epoch: 1.


In [31]:
train_loss, train_accuracy = efficientnet_sgd.evaluate(train_generator_eff)
val_loss, val_accuracy = efficientnet_sgd.evaluate(val_generator_eff)
test_loss, test_accuracy = efficientnet_sgd.evaluate(test_generator_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 185s 843ms/step - accuracy: 0.5449 - loss: 1.2430
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 342ms/step - accuracy: 0.6132 - loss: 1.0996
47/47 ━━━━━━━━━━━━━━━━━━━━ 24s 508ms/step - accuracy: 0.6015 - loss: 1.1356


In [56]:

efficientnet_sgd.save("/content/drive/MyDrive/models3/efficientnet_sgd_model.keras")

In [3]:
EfficientNetB0_data.append({
    "Model": "EfficientNetB0 SGD",
    "Train Accuracy": 54.49,
    "Validation Accuracy": 61.32,
    "Test Accuracy": 60.15,
    "Train Loss": 1.2430,
    "Validation Loss": 1.0996,
    "Test Loss": 1.1356
})

**RMSprop**

In [36]:
efficientnet_RMSprop = Model(
    inputs=base_model.input,
    outputs=output,
)

In [38]:
from tensorflow.keras.optimizers import RMSprop

efficientnet_RMSprop.compile(
    optimizer=RMSprop(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [40]:
history_efficientnet_rmsprop = efficientnet_RMSprop.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=7,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 750ms/step - accuracy: 0.6076 - loss: 1.0918 - val_accuracy: 0.6352 - val_loss: 0.9230 - learning_rate: 1.0000e-04
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 168s 765ms/step - accuracy: 0.6027 - loss: 1.1061 - val_accuracy: 0.6212 - val_loss: 0.9056 - learning_rate: 1.0000e-04
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 163s 740ms/step - accuracy: 0.6185 - loss: 1.0685 - val_accuracy: 0.6265 - val_loss: 0.9037 - learning_rate: 1.0000e-04
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 161s 732ms/step - accuracy: 0.6076 - loss: 1.0916 - val_accuracy: 0.6185 - val_loss: 0.9174 - learning_rate: 1.0000e-04
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 683ms/step - accuracy: 0.5954 - loss: 1.0358
Epoch 5: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 749ms/step - accuracy: 0.6044 - loss: 1.0716 - val_accuracy: 0.6178 - val_loss: 0.9275 - learning_rate: 1.0000e-04
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 163s 7

In [41]:
train_loss, train_accuracy = efficientnet_RMSprop.evaluate(train_generator_eff)
val_loss, val_accuracy = efficientnet_RMSprop.evaluate(val_generator_eff)
test_loss, test_accuracy = efficientnet_RMSprop.evaluate(test_generator_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 154s 701ms/step - accuracy: 0.6539 - loss: 0.8975
47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 291ms/step - accuracy: 0.6265 - loss: 0.9037
47/47 ━━━━━━━━━━━━━━━━━━━━ 21s 443ms/step - accuracy: 0.6134 - loss: 0.9606


In [42]:

efficientnet_RMSprop.save("/content/drive/MyDrive/models3/efficientnet_rmsprop.keras")

In [4]:
EfficientNetB0_data.append({
    "Model": "EfficientNetB0 RMSprop",
    "Train Accuracy": 65.39,
    "Validation Accuracy": 62.65,
    "Test Accuracy": 61.34,
    "Train Loss": 0.8975,
    "Validation Loss": 0.9037,
    "Test Loss": 0.9606
})

**Batchsize(64)**

In [43]:
train_generator_eff_64 = train_datagen_eff.flow_from_dataframe(
    dataframe=train_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=True
)

val_generator_eff_64 = val_datagen_eff.flow_from_dataframe(
    dataframe=val_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

test_generator_64 = test_datagen_eff.flow_from_dataframe(
    dataframe=test_df,
    x_col="image_path",
    y_col="dx",
    target_size=(224,224),
    batch_size=64,
    class_mode="categorical",
    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [44]:
efficientnet_64 = Model(
    inputs=base_model.input,
    outputs=output,
    name="EfficientNetB0"
)

In [45]:
efficientnet_64.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [46]:
history_efficientnet_64 = efficientnet_64.fit(
    train_generator_eff_64,
    validation_data=val_generator_eff_64,
    epochs=10,
    class_weight=class_weights,
    callbacks=[lr_scheduler, early_stop],
)

Epoch 1/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.6001 - loss: 1.0709 - val_accuracy: 0.6112 - val_loss: 0.9531 - learning_rate: 1.0000e-04
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5911 - loss: 1.1081
Epoch 2: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
110/110 ━━━━━━━━━━━━━━━━━━━━ 165s 2s/step - accuracy: 0.5790 - loss: 1.0731 - val_accuracy: 0.6032 - val_loss: 0.9516 - learning_rate: 1.0000e-04
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 164s 1s/step - accuracy: 0.5920 - loss: 1.0339 - val_accuracy: 0.6032 - val_loss: 0.9466 - learning_rate: 2.0000e-05
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [48]:
train_loss, train_accuracy = efficientnet_64.evaluate(train_generator_eff_64)
val_loss, val_accuracy = efficientnet_64.evaluate(val_generator_eff_64)
test_loss, test_accuracy = efficientnet_64.evaluate(test_generator_64)

110/110 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.6267 - loss: 0.9607
24/24 ━━━━━━━━━━━━━━━━━━━━ 16s 644ms/step - accuracy: 0.6112 - loss: 0.9531
24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 834ms/step - accuracy: 0.5935 - loss: 1.0026


In [49]:
efficientnet_64.save("/content/drive/MyDrive/models3/efficientnet_model_64.keras")

In [5]:
EfficientNetB0_data.append({
    "Model": "EfficientNetB0 batchsize(64)",
    "Train Accuracy": 62.67,
    "Validation Accuracy": 61.12,
    "Test Accuracy": 59.35,
    "Train Loss": 0.9607,
    "Validation Loss": 0.9531,
    "Test Loss": 1.0026
})

**Fine Tuning**

In [53]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(7, activation="softmax")(x)

efficientnet_ft = Model(
    inputs=base_model.input,
    outputs=output
)

efficientnet_ft.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

# Train
history_efficientnet_ft = efficientnet_ft.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)


Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 0.1173 - loss: 2.0117 - val_accuracy: 0.1052 - val_loss: 2.1518 - learning_rate: 1.0000e-05
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 222s 1s/step - accuracy: 0.1412 - loss: 1.8964 - val_accuracy: 0.1292 - val_loss: 2.0223 - learning_rate: 1.0000e-05
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 184s 835ms/step - accuracy: 0.1802 - loss: 1.8394 - val_accuracy: 0.1611 - val_loss: 1.9395 - learning_rate: 1.0000e-05
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 177s 805ms/step - accuracy: 0.2193 - loss: 1.7608 - val_accuracy: 0.1897 - val_loss: 1.8641 - learning_rate: 1.0000e-05
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 163s 740ms/step - accuracy: 0.2495 - loss: 1.7141 - val_accuracy: 0.2204 - val_loss: 1.7903 - learning_rate: 1.0000e-05
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 162s 738ms/step - accuracy: 0.2837 - loss: 1.6510 - val_accuracy: 0.2650 - val_loss: 1.7265 - learning_rate: 1.0000e-05
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 165s 7

In [54]:
train_loss, train_accuracy = efficientnet_ft.evaluate(train_generator_eff)
val_loss, val_accuracy = efficientnet_ft.evaluate(val_generator_eff)
test_loss, test_accuracy = efficientnet_ft.evaluate(test_generator_eff)

220/220 ━━━━━━━━━━━━━━━━━━━━ 161s 729ms/step - accuracy: 0.5693 - loss: 1.2568
47/47 ━━━━━━━━━━━━━━━━━━━━ 14s 305ms/step - accuracy: 0.5340 - loss: 1.2456
47/47 ━━━━━━━━━━━━━━━━━━━━ 22s 475ms/step - accuracy: 0.5077 - loss: 1.2901


In [55]:
efficientnet_ft.save("/content/drive/MyDrive/models3/efficientnet_finetuned.keras")
print("EfficientNet Fine-Tuned model saved successfully.")

EfficientNet Fine-Tuned model saved successfully.


In [6]:
EfficientNetB0_data.append({
    "Model": "EfficientNetB0 Finetuning",
    "Train Accuracy": 59.63,
    "Validation Accuracy": 53.40,
    "Test Accuracy": 50.77,
    "Train Loss": 1.2568,
    "Validation Loss": 1.2456,
    "Test Loss": 1.2901
})

In [58]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.0 MB/s eta 0:00:00


**Hyperparameter**

In [59]:
import keras_tuner as kt

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

def build_efficientnet(hp):

    base_model = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Fine-tune last layers
    base_model.trainable = True

    fine_tune = hp.Choice(
        "fine_tune_layers",
        values=[20, 30, 40]
    )

    for layer in base_model.layers[:-fine_tune]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    x = Dense(
        units=hp.Choice(
            "dense_units",
            values=[128, 256, 512]
        ),
        activation="relu"
    )(x)

    x = Dropout( hp.Choice("dropout", values=[0.3, 0.4, 0.5 ]) )(x)
    outputs = Dense(7, activation="softmax")(x)

    efficient_hyper = Model(
        inputs=base_model.input,
        outputs=outputs
    )

    optimizer = hp.Choice(
        "optimizer",
        values=["adam", "sgd", "rmsprop"]
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[1e-3, 1e-4, 1e-5]
    )

    if optimizer == "adam":
        opt = Adam(learning_rate=learning_rate)

    elif optimizer == "sgd":
        opt = SGD(
            learning_rate=learning_rate,
            momentum=0.9
        )

    else:
        opt = RMSprop(
            learning_rate=learning_rate
        )

    efficient_hyper.compile(
        optimizer=opt,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return efficient_hyper

In [62]:
tuner = kt.RandomSearch(
    build_efficientnet,
    objective="val_accuracy",
    max_trials=5,
    directory="efficientnet_tuner",
    project_name="efficientnet_hp"
)
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

Reloading Tuner from efficientnet_tuner/efficientnet_hp/tuner0.json


In [64]:
tuner.search(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr]
)

Trial 5 Complete [00h 32m 48s]
val_accuracy: 0.7017310261726379

Best val_accuracy So Far: 0.7057256698608398
Total elapsed time: 02h 17m 52s


In [65]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

{'fine_tune_layers': 40, 'dense_units': 256, 'dropout': 0.3, 'optimizer': 'sgd', 'learning_rate': 0.001}


In [66]:
best_model = tuner.get_best_models(1)[0]
best_model.save("/content/drive/MyDrive/models3/efficientnet_hyperparameter.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 2 variables whereas the saved optimizer has 42 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [67]:
train_loss, train_acc = best_model.evaluate(train_generator_eff,verbose=1)
val_loss, val_acc = best_model.evaluate( val_generator_eff,verbose=1)
test_loss, test_acc = best_model.evaluate(test_generator_eff,verbose=1)


220/220 ━━━━━━━━━━━━━━━━━━━━ 192s 811ms/step - accuracy: 0.7250 - loss: 0.7562
47/47 ━━━━━━━━━━━━━━━━━━━━ 26s 549ms/step - accuracy: 0.7057 - loss: 0.7705
47/47 ━━━━━━━━━━━━━━━━━━━━ 33s 712ms/step - accuracy: 0.6919 - loss: 0.8316


In [9]:
import pandas as pd
EfficientNetB0_data.append({
    "Model": "EfficientNetB0 HyperParameter",
    "Train Accuracy":71.48 ,
    "Validation Accuracy": 66.44,
    "Test Accuracy": 66.27,
    "Train Loss": 0.7116,
    "Validation Loss": 0.7996 ,
    "Test Loss": 0.8552
})

In [10]:
comparison_efficient_datas = pd.DataFrame(EfficientNetB0_data)
comparison_efficient_datas

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,EfficientNetB0,51.68,56.26,54.49,1.3004,1.2470,1.2937
1,EfficientNetB0 ES_LR,61.95,59.99,60.55,1.1024,1.0835,1.1392
2,EfficientNetB0 SGD,54.49,61.32,60.15,1.2430,1.0996,1.1356
3,EfficientNetB0 RMSprop,65.39,62.65,61.34,0.8975,0.9037,0.9606
4,EfficientNetB0 batchsize(64),62.67,61.12,59.35,0.9607,0.9531,1.0026
5,EfficientNetB0 Finetuning,59.63,53.40,50.77,1.2568,1.2456,1.2901
6,EfficientNetB0 HyperParameter,71.48,66.44,66.27,0.7116,0.7996,0.8552
7,EfficientNetB0 HyperParameter,71.48,66.44,66.27,0.7116,0.7996,0.8552


In [11]:
comparison_efficient_datas.to_csv("models2/efficient_model_comparison.csv", index=False)
print("CSV saved successfully!")

CSV saved successfully!
